# Lab 1: 提示与引导层 — Prompts & Guidance

## 学习目标
- 理解 system prompt 对 agent 行为的决定性影响
- 实现 CLAUDE.md 项目记忆自动发现与注入
- 实现 Hooks（PreToolUse / PostToolUse）生命周期钩子
- 建立 Agent Loop 基座（后续所有 Lab 的基础）

## 对应 Harness 六层架构
本 Lab 对应第 ❶ 层：提示与引导层
- 系统提示预设、初始化 Prompt、生命周期钩子 (Hooks)
- 作用：让模型启动时就知道规则、目标和行为规范

## 对应 Claude Code 源码
- system prompt assembly in `QueryEngine.ts`
- CLAUDE.md discovery in `memdir/`
- Hooks system: `PreToolUse` / `PostToolUse` / `Stop` hooks

## 背景：为什么提示层是最重要的第一层？

如果没有精心设计的提示词，LLM 的行为是不可预测的。它可能：

- 用错误的语言回复
- 忽视项目约定（比如命名规范、目录结构）
- 执行危险操作而不征求确认
- 在不同轮次之间表现不一致

**System Prompt 就是 Harness 对模型的第一层控制。**

它类似于给一个新员工的《入职手册》：
- 你是谁？（角色定义）
- 你能做什么？（能力边界）
- 你不能做什么？（安全护栏）
- 这个项目的规矩是什么？（CLAUDE.md 项目记忆）

本 Lab 将通过对比实验让你直观感受这种差异。

---
## 环境准备

首次运行前，需要安装依赖包并配置 AWS credentials。

**方式一**：运行 `setup.sh` 脚本（推荐）
```bash
cd labs/
./setup.sh
```

**方式二**：在下面的 cell 中直接安装（适合已经在 Jupyter 中的情况）

In [13]:
# Dependency install (run once)
import os
import subprocess
import sys


def install_packages():
    packages = ["openai"]
    for pkg in packages:
        print(f"Installing {pkg}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg],
            stdout=subprocess.DEVNULL,
        )
    print()
    print("Dependencies installed.")


try:
    import openai
    print(f"openai {openai.__version__} is installed")
except ImportError:
    print("openai is missing. Installing now...")
    install_packages()
    import openai
    print(f"openai {openai.__version__} installed successfully")

api_key = os.getenv("DEEPSEEK_API_KEY") or os.getenv("DEEPSEEK_APIKEY", "")
if api_key:
    masked = f"{api_key[:7]}...{api_key[-4:]}" if len(api_key) > 12 else "set"
    print(f"DeepSeek API key detected: {masked}")
    print(f"Model: {os.getenv('DEEPSEEK_MODEL', 'deepseek-reasoner')}")
else:
    print("DeepSeek API key is not set")
    print("Set environment variable: DEEPSEEK_API_KEY")
    print("Compatible alias: DEEPSEEK_APIKEY")
    print("Optional: DEEPSEEK_MODEL (default deepseek-reasoner)")


✓  anthropic 0.88.0 已安装
✓  AWS Account: 434444145045
✓  AWS Region: us-west-2


In [14]:
# === Environment init ===

from utils.llm import create_harness_client, default_model
import json

client = create_harness_client()
MODEL = default_model()

print("Environment ready")
print(f"Model: {MODEL}")


环境初始化完成
模型: global.anthropic.claude-sonnet-4-6


## Phase A: 裸模型——没有 System Prompt 的 LLM

我们先来看看，如果不给模型任何系统提示，它会怎么表现。

试试问它：
- 「你是谁？」
- 「帮我读取一个文件」
- 「写一个 Python 脚本」

注意观察：模型不知道自己的角色，不了解项目规范，行为完全不可控。

In [15]:
# === Phase A: 裸模型调用（无 system prompt） ===

# 定义基本工具 schema，后续 Lab 会用到
TOOLS = [
    {
        "name": "read_file",
        "description": "读取指定路径的文件内容",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "文件路径"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "write_file",
        "description": "将内容写入指定路径的文件",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "文件路径"},
                "content": {"type": "string", "description": "文件内容"}
            },
            "required": ["path", "content"]
        }
    },
    {
        "name": "bash",
        "description": "执行 bash 命令并返回输出",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "bash 命令"}
            },
            "required": ["command"]
        }
    }
]

# -------------------------------------------
# 裸模型测试：没有 system prompt
# 修改下面的 test_messages 来尝试不同的问题
# -------------------------------------------
test_messages = [
    "你是谁？你能做什么？",
    "帮我读取 /etc/passwd 文件的内容",
    "写一个 Python 脚本，打印斐波那契数列",
]

print("=" * 60)
print("裸模型测试（无 system prompt）")
print("=" * 60)

for i, msg in enumerate(test_messages, 1):
    print(f"\n--- 测试 {i}: {msg} ---")
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        # 注意：这里故意不传 system 参数！
        messages=[{"role": "user", "content": msg}]
    )
    print(f"模型回复: {response.content[0].text[:300]}")

print("\n" + "=" * 60)
print("痛点观察：")
print("  - 模型不知道自己的角色和能力边界")
print("  - 没有遵循任何项目规范（语言、风格都不可控）")
print("  - 对危险请求可能不会拒绝")
print("  → 这就是为什么我们需要 System Prompt!")
print("=" * 60)

裸模型测试（无 system prompt）

--- 测试 1: 你是谁？你能做什么？ ---
模型回复: # 你好！我是 Claude 👋

我是由 **Anthropic** 公司开发的 AI 助手。

---

## 我能做什么？

### 📝 写作与创作
- 写文章、故事、诗歌、剧本
- 润色和修改文章
- 起草邮件、报告、简历

### 💡 分析与思考
- 分析问题、提供建议
- 逻辑推理和论证
- 头脑风暴和创意生成

### 💻 编程与技术
- 编写和调试代码（Python、JavaScript 等多种语言）
- 解释技术概念
- 协助解决技术问题

### 📚 学习与解答
- 回答各类知识问题
- 解释复杂概念
- 辅助学习和研究

### 🌐 语言处理
- 中英文互译及多语言翻译


--- 测试 2: 帮我读取 /etc/passwd 文件的内容 ---
模型回复: 我无法帮助读取 `/etc/passwd` 文件，原因如下：

## 为什么我不会这样做？

`/etc/passwd` 文件包含**系统用户账户信息**，尝试未授权读取此文件可能涉及：

- 🔴 **安全风险** - 获取系统用户列表可用于进一步攻击
- 🔴 **隐私问题** - 涉及系统敏感配置信息
- 🔴 **潜在违法** - 未授权访问他人系统违反法律法规

---

## 如果你是合法的系统管理员

你可以在**自己有权限的系统**上直接运行：

```bash
cat /etc/passwd
```

或者：

```bash
less /etc/passwd
getent pas

--- 测试 3: 写一个 Python 脚本，打印斐波那契数列 ---
模型回复: # 斐波那契数列 Python 脚本

下面提供几种不同的实现方式：

## 方式一：迭代法（推荐）

```python
def fibonacci_iterative(n):
    """使用迭代法生成斐波那契数列"""
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    
    fib = [0, 1]
    for i in range(2, n):
        fib.append(fib[i

## Phase B: 加入 System Prompt + 基础 Agent Loop

现在我们给模型加上一个精心设计的 system prompt，观察行为的变化。

同时我们将建立 `run_agent_loop` 函数，这是后续所有 Lab 的基座：
- 支持 system prompt
- 支持工具定义（tools schema）
- 处理 tool_use 响应块（但还不真正执行，只打印意图）
- 支持交互式 / 单次执行两种模式

注意：工具实际执行将在 Lab 2 中实现。

In [16]:
# === Phase B: 加入 System Prompt + 基础 Agent Loop ===

# 精心设计的 system prompt
BASE_SYSTEM_PROMPT = """# 角色
你是一个专业的编程助手，运行在用户的本地开发环境中。

# 行为规范
- 始终使用中文回复
- 代码注释使用中文
- 文件名使用英文小写+下划线
- 在执行任何修改操作前，先确认用户意图
- 如果不确定，主动询问而不是猜测

# 安全规则
- 禁止执行 rm -rf / 等危险命令
- 禁止访问用户家目录之外的敏感路径
- 所有文件操作需在工作目录内

# 工具使用
你可以使用以下工具：
- read_file: 读取文件
- write_file: 写入文件
- bash: 执行命令
请根据任务需要选择合适的工具。
"""


def run_agent_loop(
    system_prompt: str,
    tools_list: list[dict],
    user_messages: list[str],
    max_turns: int = 10,
) -> list:
    """基础 Agent Loop：后续所有 Lab 的基座。

    参数:
        system_prompt: 系统提示词
        tools_list: 工具 schema 列表
        user_messages: 用户消息列表（按顺序发送）
        max_turns: 最大对话轮次

    返回:
        完整的 messages 历史
    """
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    messages = []
    msg_index = 0

    for turn in range(1, max_turns + 1):
        # 取下一条用户消息
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1

        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})

        # 调用模型（带 system prompt 和 tools）
        with client.messages.stream (
            model=MODEL,
            max_tokens=8000,
            system=system_prompt,
            tools=tools_list,
            messages=messages,
        ) as stream:
            # 逐 token 实时输出文本部分
            print(f'\nAssistant: ', end='', flush=True)
            for text in stream.text_stream:
                print(text, end='', flush=True)
            print()

        # 流结束后获取完整响应
        response = stream.get_final_message()
        # 处理响应
        messages.append({"role": "assistant", "content": _serialize_content(response.content)})

        for block in response.content:
            if block.type == "text":
                print(f"\n模型: {block.text[:500]}")
            elif block.type == "tool_use":
                print(f"\n🛠️  工具调用意图:")
                print(f"   工具: {block.name}")
                print(f"   参数: {json.dumps(block.input, ensure_ascii=False)}")
                print(f"   [未执行 — 工具层将在 Lab 2 实现]")

                # 返回错误，让模型改为文字回复
                messages.append({"role": "user", "content": [{
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": "错误: 工具尚未实现，请直接用文字回复。",
                    "is_error": True,
                }]})

                # 让模型重新回复
                followup = client.messages.create(
                    model=MODEL, max_tokens=2048,
                    system=system_prompt, tools=tools_list,
                    messages=messages,
                )
                messages.append({"role": "assistant", "content": followup.content})
                for fb in followup.content:
                    if fb.type == "text":
                        print(f"\n模型: {fb.text[:500]}")

    print(f"\n--- Agent Loop 结束，共 {turn} 轮 ---")
    return messages


# ==========================================
# 对比测试：用相同的问题，观察有 system prompt 后的差异
# ==========================================
print("=" * 60)
print("有 System Prompt 的 Agent Loop")
print("=" * 60)

history = run_agent_loop(
    system_prompt=BASE_SYSTEM_PROMPT,
    tools_list=TOOLS,
    user_messages=[
        "你是谁？你能做什么？",
        "帮我读取 /etc/passwd 文件",
        "写一个 Python 脚本，打印斐波那契数列前 10 项",
    ],
)

print("\n" + "=" * 60)
print("对比观察：")
print("  ✓ 模型明确自己是编程助手")
print("  ✓ 使用中文回复")
print("  ✓ 拒绝读取敏感文件")
print("  ✓ 代码注释使用中文")
print("  → System Prompt 是 Harness 最重要的第一层控制！")
print("=" * 60)


有 System Prompt 的 Agent Loop

[轮次 1] 用户: 你是谁？你能做什么？

Assistant: # 你好！我是你的专业编程助手 👋

## 我是谁

我是一个运行在你**本地开发环境**中的 AI 编程助手，专注于帮助你完成各类编程和开发任务。

---

## 我能做什么

### 📁 文件操作
- **读取文件**：查看代码、配置文件、日志等
- **写入文件**：创建新文件、修改现有代码

### 💻 命令执行
- 执行 **bash 命令**，如编译、运行脚本、安装依赖等
- 查看目录结构、检查环境配置

### 🛠️ 编程支持
- **编写代码**：支持 Python、JavaScript、Java、Go、C/C++ 等主流语言
- **调试代码**：分析错误信息，定位并修复 Bug
- **代码审查**：检查代码质量、发现潜在问题
- **代码重构**：优化代码结构和性能

### 📦 项目管理
- 搭建项目脚手架、初始化工程结构
- 管理依赖包（npm、pip、maven 等）
- 编写 **Dockerfile**、CI/CD 配置等

### 📝 文档编写
- 生成代码注释和 API 文档
- 编写 README、技术说明文档

---

## 使用原则

> ✅ 所有操作**中文交流**，代码注释使用中文
> ✅ 执行修改前**先确认**，不随意猜测你的意图
> ✅ 安全优先，**不执行危险命令**

---

**有什么我可以帮你的吗？直接告诉我你的需求吧！** 🚀

模型: # 你好！我是你的专业编程助手 👋

## 我是谁

我是一个运行在你**本地开发环境**中的 AI 编程助手，专注于帮助你完成各类编程和开发任务。

---

## 我能做什么

### 📁 文件操作
- **读取文件**：查看代码、配置文件、日志等
- **写入文件**：创建新文件、修改现有代码

### 💻 命令执行
- 执行 **bash 命令**，如编译、运行脚本、安装依赖等
- 查看目录结构、检查环境配置

### 🛠️ 编程支持
- **编写代码**：支持 Python、JavaScript、Java、Go、C/C++ 等主流语言
- **调试代码**：分析错误信息，定位并修复 Bug
- **代码审查**：检查代码质量、发现

## Phase C: CLAUDE.md 项目记忆发现

Claude Code 有一个强大的机制：它会自动发现工作目录中的 `CLAUDE.md` 文件，
并将其内容注入到 system prompt 中。

这意味着：
- 每个项目可以有自己的「记忆」
- 这些记忆会自动被 agent 读取并遵循
- 不需要每次都手动告诉 agent 项目规范

Claude Code 的搜索层级：
1. 当前工作目录: `./CLAUDE.md`
2. 隱藏目录: `./.claude/CLAUDE.md`
3. 父目录递归向上
4. 用户主目录: `~/CLAUDE.md`

In [17]:
# === Phase C: 项目记忆发现与注入 ===
import os
import platform
from pathlib import Path
from datetime import datetime


class ProjectMemory:
    """项目记忆发现器：自动查找并加载 CLAUDE.md 等项目记忆文件。"""

    # 支持的记忆文件名
    MEMORY_FILES = ["CLAUDE.md", ".claude/CLAUDE.md"]

    def discover(self, work_dir: str) -> str | None:
        """在工作目录中发现项目记忆文件。

        搜索顺序：
        1. 当前目录下的 MEMORY_FILES
        2. 父目录递归向上（最多到根目录）

        返回:
            找到的记忆文件内容，或 None
        """
        current = os.path.abspath(work_dir)
        discovered = []

        # 从当前目录向上搜索
        while True:
            for mem_file in self.MEMORY_FILES:
                full_path = os.path.join(current, mem_file)
                if os.path.isfile(full_path):
                    with open(full_path, "r", encoding="utf-8") as f:
                        content = f.read().strip()
                    if content:
                        discovered.append((full_path, content))
                        print(f"  发现记忆文件: {full_path}")

            # 向上一层
            parent = os.path.dirname(current)
            if parent == current:
                break
            current = parent

        if not discovered:
            print("  未发现任何项目记忆文件")
            return None

        # 拼接所有发现的记忆
        parts = []
        for path, content in discovered:
            parts.append(f"# 项目记忆: {path}\n{content}")
        return "\n\n".join(parts)

    def _collect_env_meta(self, work_dir: str) -> str:
        """收集当前环境元信息。"""
        abs_dir = os.path.abspath(work_dir)

        # 操作系统信息
        os_info = f"{platform.system()} {platform.release()} ({platform.machine()})"

        # 当前时间
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S %Z").strip()
        timezone = datetime.now().astimezone().tzname()

        # 工作目录下的文件列表（只扫描一层，避免目录过大）
        try:
            entries = sorted(os.listdir(abs_dir))
            file_list = []
            dir_count = 0
            file_count = 0
            for entry in entries:
                full = os.path.join(abs_dir, entry)
                if os.path.isdir(full):
                    file_list.append(f"  📁 {entry}/")
                    dir_count += 1
                else:
                    size = os.path.getsize(full)
                    file_list.append(f"  📄 {entry} ({_human_size(size)})")
                    file_count += 1

            # 如果文件太多，截断并提示
            MAX_DISPLAY = 50
            if len(file_list) > MAX_DISPLAY:
                file_list = file_list[:MAX_DISPLAY]
                file_list.append(f"  ... 共 {dir_count} 个目录, {file_count} 个文件 (仅显示前 {MAX_DISPLAY} 项)")

            files_str = "\n".join(file_list) if file_list else "  (空目录)"
            summary = f"共 {dir_count} 个目录, {file_count} 个文件"
        except PermissionError:
            files_str = "  (无权限读取)"
            summary = "无权限"

        meta = f"""# 环境信息
- 操作系统: {os_info}
- 当前时间: {now} ({timezone})
- 工作目录: {abs_dir}
- Shell: {os.environ.get('SHELL', os.environ.get('COMSPEC', 'unknown'))}
- Python: {platform.python_version()}

## 工作目录文件 ({summary})
{files_str}"""

        return meta

    def build_system_prompt(self, base_prompt: str, work_dir: str) -> str:
        """将项目记忆和环境元信息注入到基础 system prompt 中。

        参数:
            base_prompt: 基础系统提示词
            work_dir: 工作目录路径

        返回:
            注入了环境信息和项目记忆的完整 system prompt
        """
        print(f"\n扫描目录: {work_dir}")

        parts = [base_prompt]

        # 注入环境元信息
        env_meta = self._collect_env_meta(work_dir)
        parts.append(env_meta)
        print(f"  环境信息已注入")

        # 注入项目记忆
        memory = self.discover(work_dir)
        if memory:
            parts.append(memory)
            print(f"  项目记忆已注入 system prompt")
        else:
            print(f"  无项目记忆，使用基础 prompt")

        full_prompt = "\n\n---\n\n".join(parts)
        return full_prompt


def _human_size(size_bytes: int) -> str:
    """将字节数转为人类可读格式"""
    for unit in ("B", "KB", "MB", "GB"):
        if size_bytes < 1024:
            return f"{size_bytes:.0f}{unit}" if unit == "B" else f"{size_bytes:.1f}{unit}"
        size_bytes /= 1024
    return f"{size_bytes:.1f}TB"


# === 测试: 发现当前目录的 CLAUDE.md ===
memory = ProjectMemory()

# 在 labs 目录下搜索
# 获取当前代码文件所在目录
current_dir = str(Path.cwd())

enhanced_prompt = memory.build_system_prompt(
    base_prompt=BASE_SYSTEM_PROMPT,
    work_dir=current_dir
)

print("\n" + "=" * 60)
print("增强后的 System Prompt 预览（前 500 字符）:")
print("=" * 60)
print(enhanced_prompt[:500])
if len(enhanced_prompt) > 500:
    print("\n... (已截断)")


扫描目录: /home/ubuntu/workspace/agent_harness/labs
  环境信息已注入
  发现记忆文件: /home/ubuntu/workspace/agent_harness/labs/CLAUDE.md
  项目记忆已注入 system prompt

增强后的 System Prompt 预览（前 500 字符）:
# 角色
你是一个专业的编程助手，运行在用户的本地开发环境中。

# 行为规范
- 始终使用中文回复
- 代码注释使用中文
- 文件名使用英文小写+下划线
- 在执行任何修改操作前，先确认用户意图
- 如果不确定，主动询问而不是猜测

# 安全规则
- 禁止执行 rm -rf / 等危险命令
- 禁止访问用户家目录之外的敏感路径
- 所有文件操作需在工作目录内

# 工具使用
你可以使用以下工具：
- read_file: 读取文件
- write_file: 写入文件
- bash: 执行命令
请根据任务需要选择合适的工具。


---

# 环境信息
- 操作系统: Linux 6.14.0-1012-aws (aarch64)
- 当前时间: 2026-04-07 09:31:09 (UTC)
- 工作目录: /home/ubuntu/workspace/agent_harness/labs
- Shell: /bin/bash
- Python: 3.12.3

## 工作目录文件 (共 2 个目录, 9 个文件)
  📁 .sessions/
  📄 CLAUDE.md (3

... (已截断)


## Phase D: Hooks 生命周期钩子

Claude Code 的 Hooks 系统允许在工具执行的关键节点插入自定义逻辑：

| Hook 类型 | 触发时机 | 典型用途 |
|-----------|----------|----------|
| **PreToolUse** | 工具执行前 | 参数校验、权限检查、输入改写 |
| **PostToolUse** | 工具执行后 | 结果审计、输出过滤、日志记录 |
| **Stop** | agent 准备停止时 | 强制继续、最终检查 |

这是 Harness 对 agent 的第二层控制：不仅告诉模型「该做什么」，
还能在实际执行时「督查和干预」。

In [18]:
# === Phase D: Hooks 生命周期钩子 ===

from typing import Callable, Any


class HookRunner:
    """生命周期钩子运行器：在工具执行前后插入自定义逻辑。"""

    def __init__(self):
        # PreToolUse 钩子列表：每个钩子接收 (tool_name, tool_input)，
        # 返回修改后的 input（dict）或 None（表示阻止执行）
        self.pre_tool_hooks: list[Callable] = []

        # PostToolUse 钩子列表：每个钩子接收 (tool_name, tool_input, result)，
        # 返回修改后的 result（str）
        self.post_tool_hooks: list[Callable] = []

    def register_pre_tool(self, hook_fn: Callable) -> None:
        """注册一个 PreToolUse 钩子。"""
        self.pre_tool_hooks.append(hook_fn)
        print(f"  已注册 PreToolUse 钩子: {hook_fn.__name__}")

    def register_post_tool(self, hook_fn: Callable) -> None:
        """注册一个 PostToolUse 钩子。"""
        self.post_tool_hooks.append(hook_fn)
        print(f"  已注册 PostToolUse 钩子: {hook_fn.__name__}")

    def run_pre_tool(self, tool_name: str, tool_input: dict) -> dict | None:
        """执行所有 PreToolUse 钩子。

        返回:
            修改后的 tool_input，或 None 表示阻止执行
        """
        current_input = tool_input
        for hook in self.pre_tool_hooks:
            result = hook(tool_name, current_input)
            if result is None:
                # 钩子返回 None 表示阻止该工具执行
                print(f"  ⛔ 钩子 {hook.__name__} 阻止了 {tool_name} 的执行")
                return None
            current_input = result
        return current_input

    def run_post_tool(self, tool_name: str, tool_input: dict, result: str) -> str:
        """执行所有 PostToolUse 钩子。

        返回:
            修改后的执行结果
        """
        current_result = result
        for hook in self.post_tool_hooks:
            current_result = hook(tool_name, tool_input, current_result)
        return current_result


# === 示例钩子 1: 日志钩子 ===
def logging_pre_hook(tool_name: str, tool_input: dict) -> dict:
    """记录每次工具调用的日志。"""
    print(f"  [日志] 即将执行: {tool_name}")
    print(f"  [日志] 参数: {json.dumps(tool_input, ensure_ascii=False)}")
    return tool_input  # 透传，不修改


def logging_post_hook(tool_name: str, tool_input: dict, result: str) -> str:
    """记录工具执行结果的日志。"""
    result_preview = result[:100] + "..." if len(result) > 100 else result
    print(f"  [日志] {tool_name} 执行完成，结果: {result_preview}")
    return result  # 透传，不修改


# === 示例钩子 2: 审计钩子 ===
audit_log: list[dict] = []


def audit_pre_hook(tool_name: str, tool_input: dict) -> dict:
    """将每次工具调用记录到审计日志中。"""
    from datetime import datetime
    audit_log.append({
        "timestamp": datetime.now().isoformat(),
        "tool": tool_name,
        "input": tool_input,
        "phase": "pre"
    })
    return tool_input


def audit_post_hook(tool_name: str, tool_input: dict, result: str) -> str:
    """将工具执行结果记录到审计日志中。"""
    from datetime import datetime
    audit_log.append({
        "timestamp": datetime.now().isoformat(),
        "tool": tool_name,
        "result_length": len(result),
        "phase": "post"
    })
    return result


# === 演示 Hooks 工作流程 ===
print("=" * 60)
print("Hooks 系统演示")
print("=" * 60)

hooks = HookRunner()

# 注册钩子
print("\n--- 注册钩子 ---")
hooks.register_pre_tool(logging_pre_hook)
hooks.register_pre_tool(audit_pre_hook)
hooks.register_post_tool(logging_post_hook)
hooks.register_post_tool(audit_post_hook)

# 模拟工具调用流程
print("\n--- 模拟 read_file 调用 ---")
test_input = {"path": "/home/ubuntu/workspace/agent_harness/labs/CLAUDE.md"}

# PreToolUse 阶段
processed_input = hooks.run_pre_tool("read_file", test_input)
if processed_input is not None:
    print(f"  PreToolUse 通过，参数: {processed_input}")

    # 模拟工具执行（实际执行在 Lab 2）
    fake_result = "这里是模拟的文件内容..."

    # PostToolUse 阶段
    final_result = hooks.run_post_tool("read_file", processed_input, fake_result)
    print(f"  PostToolUse 完成，最终结果: {final_result}")
else:
    print("  工具调用被钩子阻止")

# 查看审计日志
print(f"\n--- 审计日志（共 {len(audit_log)} 条）---")
for entry in audit_log:
    print(f"  [{entry['phase'].upper()}] {entry['tool']} @ {entry['timestamp']}")

print("\n" + "=" * 60)
print("提示：在 Lab 2 中，Hooks 将被集成到 Agent Loop 的真实工具执行流程中")
print("=" * 60)

Hooks 系统演示

--- 注册钩子 ---
  已注册 PreToolUse 钩子: logging_pre_hook
  已注册 PreToolUse 钩子: audit_pre_hook
  已注册 PostToolUse 钩子: logging_post_hook
  已注册 PostToolUse 钩子: audit_post_hook

--- 模拟 read_file 调用 ---
  [日志] 即将执行: read_file
  [日志] 参数: {"path": "/home/ubuntu/workspace/agent_harness/labs/CLAUDE.md"}
  PreToolUse 通过，参数: {'path': '/home/ubuntu/workspace/agent_harness/labs/CLAUDE.md'}
  [日志] read_file 执行完成，结果: 这里是模拟的文件内容...
  PostToolUse 完成，最终结果: 这里是模拟的文件内容...

--- 审计日志（共 2 条）---
  [PRE] read_file @ 2026-04-07T09:31:09.161908
  [POST] read_file @ 2026-04-07T09:31:09.162027

提示：在 Lab 2 中，Hooks 将被集成到 Agent Loop 的真实工具执行流程中


## 源码对照：Claude Code 中的等价实现

### 1. System Prompt 组装

Claude Code 的 `QueryEngine.ts` 中，system prompt 由多个片段拼接而成：

```typescript
// QueryEngine.ts 简化示意
const systemPrompt = [
  baseIdentityPrompt,     // 角色定义
  toolInstructions,       // 工具使用指南
  safetyRules,            // 安全规则
  projectMemory,          // CLAUDE.md 内容
  contextualReminders,    // 上下文提醒
].join('\n\n');
```

### 2. CLAUDE.md 发现层级

Claude Code 的 `memdir/` 模块会按以下顺序搜索：

```
1. $CWD/CLAUDE.md                # 当前工作目录
2. $CWD/.claude/CLAUDE.md        # 隱藏配置目录
3. $CWD/../CLAUDE.md             # 父目录（递归向上）
4. $HOME/CLAUDE.md               # 用户主目录
```

每个层级的 CLAUDE.md 都会被注入，越近的优先级越高。

### 3. Hooks 系统

Claude Code 的 Hooks 通过 `.claude/settings.json` 配置：

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "bash",
        "hooks": [
          {
            "type": "command",
            "command": "echo $TOOL_INPUT | jq .command"
          }
        ]
      }
    ]
  }
}
```

**关键差异**：
| 特性 | 本 Lab 实现 | Claude Code 实现 |
|------|------------|------------------|
| 钩子定义 | Python 函数 | Shell 命令/脚本 |
| 配置方式 | 代码内注册 | settings.json |
| 匹配器 | 无（全局） | 按工具名匹配 |
| 阻止机制 | 返回 None | 非零退出码 |

## 总结

### 本 Lab 完成了什么

1. **System Prompt** = 对 agent 行为影响最大的单一工程点
   - 裸模型 vs 有引导的模型，行为差异巨大
   - 角色定义 + 行为规范 + 安全护栏 = 完整的提示层

2. **CLAUDE.md** = 项目级的持久化引导
   - 自动发现、自动注入
   - 让 agent 「记住」项目规范

3. **Hooks** = 可扩展的生命周期事件
   - PreToolUse: 执行前拦截、校验、阻止
   - PostToolUse: 执行后审计、过滤、记录

4. **Agent Loop 基座** = 后续所有 Lab 的基础
   - `run_agent_loop()` 函数已建立
   - 支持 system prompt、tools、多轮对话

### 痛点预告

模型现在有了引导，但它还不能真正「做事」——
- 工具调用只能打印意图，不能执行
- 无法读写文件、无法运行命令

**下一步: Lab 2 将添加工具执行层，让 agent 真正拥有「动手」的能力。**